# نظام RAG التقليدي المتكامل باستخدام ChromaDB (الأجزاء 1، 2، 3)
### Building Complete Traditional RAG Pipeline with ChromaDB & LangChain

دليل عملي شامل ومبسط يغطي **المحاضرات 25 و 26 و 27**:
1. **المحاضرة 25**: قراءة المستندات بـ `DirectoryLoader` وتجزئتها بـ `RecursiveCharacterTextSplitter`.
2. **المحاضرة 26**: توليد التضمينات وحفظها محلياً على القرص في `ChromaDB` والبحث بالمسافات ($L_2$).
3. **المحاضرة 27**: تحويل الفهرس إلى `Retriever`، تصميم قالب التوجيه، وبناء سلسلة `create_retrieval_chain` مع الـ LLM لتوليد الإجابات النهائية المعتمدة على السياق.

In [ ]:
import os
import shutil
from dotenv import load_dotenv

# استيرادات LangChain القياسية الحديثة
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents.stuff import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

load_dotenv()
print("تم تحميل كافة المكتبات بنجاح!")

--- 
## 1. تجهيز وقراءة المستندات النصية (Ingestion)

In [ ]:
# 3 وثائق تجريبية في مجالات الذكاء الاصطناعي
docs = {
    "document_0.txt": (
        "Machine learning is a subset of artificial intelligence that learns from data without explicit programming. "
        "Supervised learning uses labeled datasets with algorithms like regression, decision trees, and support vector machines. "
        "Unsupervised learning finds hidden patterns in unlabeled data using clustering techniques."
    ),
    "document_1.txt": (
        "Deep learning is a specialized branch of machine learning inspired by artificial neural networks with multiple layers. "
        "Convolutional Neural Networks (CNNs) are best used for computer vision and image processing tasks. "
        "Recurrent Neural Networks (RNNs) and LSTMs excel at sequential and time-series data. Training relies on backpropagation."
    ),
    "document_2.txt": (
        "Natural language processing (NLP) enables machines to understand, interpret, and generate human language. "
        "The Transformer architecture revolutionized NLP in 2017 with self-attention mechanisms, replacing older recurrence. "
        "Pre-trained models like BERT and GPT set state-of-the-art benchmarks in translation, summarization, and question answering."
    )
}

sample_dir = "./chroma_sample_data"
os.makedirs(sample_dir, exist_ok=True)

for filename, content in docs.items():
    with open(os.path.join(sample_dir, filename), "w", encoding="utf-8") as f:
        f.write(content)

# قراءة الملفات دفعة واحدة
loader = DirectoryLoader(sample_dir, glob="*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
raw_documents = loader.load()
print(f"تم تحميل {len(raw_documents)} مستندات بنجاح من المجلد: {sample_dir}")

--- 
## 2. تقطيع النصوص إلى Chunks وتوليد التضمينات (Splitting & Embeddings)

In [ ]:
# تقطيع النصوص إلى قطع صغيرة بطول 250 حرف وتداخل 30 حرف
splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=30)
chunks = splitter.split_documents(raw_documents)
print(f"عدد الـ Chunks الناتجة: {len(chunks)}\n")

# إعداد نموذج التضمين (OpenAI أو FastEmbed المحلي فائق السرعة مجاناً)
openai_key = os.getenv("OPENAI_API_KEY", "").strip()
if openai_key and not openai_key.startswith("your_"):
    print("نموذج التضمين: OpenAI (text-embedding-3-small)")
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
else:
    print("نموذج التضمين: FastEmbed المحلي (BAAI/bge-small-en-v1.5) - يعمل محلياً وسريع ومجاني")
    from langchain_community.embeddings import FastEmbedEmbeddings
    embeddings = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")

--- 
## 3. الفهرسة والتخزين الدائم في ChromaDB (Persistence)

In [ ]:
persist_dir = "./my_chroma_db"
if os.path.exists(persist_dir):
    shutil.rmtree(persist_dir, ignore_errors=True)

# بناء وحفظ الفهرس محلياً على القرص
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_dir,
    collection_name="rag_collection"
)
print(f"تم حفظ {len(chunks)} متجهات بنجاح في: {persist_dir}")

--- 
## 4. تحويل الفهرس إلى مسترجع (Retriever)
الـ `Retriever` هو الواجهة البرمجية الموحدة التي تسمح لـ LangChain بالبحث في الفهرس وتمرير النتائج لسلاسل التنفيذ.

In [ ]:
# تحويل VectorStore إلى Retriever يسترجع أفضل 3 نتائج (k=3)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("تم إنشاء الـ Retriever بنجاح")

--- 
## 5. تهيئة نموذج اللغة الكبير (LLM)
نستخدم `ChatOpenAI(model="gpt-3.5-turbo")` أو نموذج بديل مدعوم.

In [ ]:
def get_llm():
    if openai_key and not openai_key.startswith("your_"):
        print("استخدام نموذج ChatOpenAI (gpt-3.5-turbo)")
        return ChatOpenAI(model="gpt-3.5-turbo", temperature=0.2, max_tokens=500)
    groq_key = os.getenv("GROQ_API_KEY", "").strip()
    if groq_key and not groq_key.startswith("your_"):
        print("استخدام نموذج ChatGroq (llama-3.1-8b-instant)")
        from langchain_groq import ChatGroq
        return ChatGroq(model="llama-3.1-8b-instant", temperature=0.2)
    # محاكي تعليمي للتشغيل بدون مفتاح خارجي
    print("ملاحظة: لم يتم العثور على مفتاح OpenAI/Groq، سيتم استخدام محاكي ذكي للتحقق من السلسلة")
    from langchain_core.language_models.fake_chat_models import FakeListChatModel
    return FakeListChatModel(responses=[
        "Deep learning is a specialized branch of machine learning based on multi-layer neural networks inspired by the human brain.",
        "Convolutional Neural Networks (CNNs) are best used for computer vision and image processing tasks.",
        "Supervised learning and Unsupervised learning are the main types of machine learning."
    ])

llm = get_llm()

--- 
## 6. تصميم قالب التوجيه (Prompt Template)
نحدد تعليمات صارمة للـ LLM بالاعتماد فقط على السياق المسترجع (`context`) دون اختلاق معلومات.

In [ ]:
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

print("تم إنشاء قالب التوجيه بنجاح!")

--- 
## 7. بناء سلسلة الـ RAG المتكاملة (`create_retrieval_chain`)
- **`create_stuff_documents_chain`**: تجمع وثائق السياق المسترجعة وتضعها في حقل `{context}` داخل القالب.
- **`create_retrieval_chain`**: تربط الـ Retriever بسلسلة المستندات لإدارة التدفق من سؤال المستخدم حتى الجواب.

In [ ]:
# 1. دمج المستندات داخل القالب
document_chain = create_stuff_documents_chain(llm, prompt)

# 2. بناء السلسلة الكاملة المتصلة بالمسترجع
rag_chain = create_retrieval_chain(retriever, document_chain)
print("تم بناء خط أنابيب الـ RAG بنجاح!")

--- 
## 8. اختبار وتشغيل النظام (Invoking the RAG Pipeline)

In [ ]:
query = "What is deep learning?"
response = rag_chain.invoke({"input": query})

print(f"السؤال: {query}\n")
print(f"الإجابة التوليدية (LLM Answer):\n{response['answer']}\n")
print("السياق المسترجع (Retrieved Context Chunks):")
for idx, doc in enumerate(response["context"], start=1):
    src = os.path.basename(doc.metadata.get("source", ""))
    print(f"  [{idx}] ({src}): {doc.page_content[:90]}...")

--- 
## 9. دالة مساعدة لتجربة أسئلة متعددة

In [ ]:
def ask_rag(question: str):
    print(f"\n{'='*60}")
    print(f"❓ السؤال: {question}")
    res = rag_chain.invoke({"input": question})
    print(f"💬 الإجابة: {res['answer']}")
    print("📚 المصادر:")
    for d in res["context"]:
        print(f"   - {os.path.basename(d.metadata.get('source', ''))}: {d.page_content[:80]}...")

# تجربة عدة استعلامات
ask_rag("What are CNNs best used for?")
ask_rag("What are the main types of machine learning?")

## 🎯 ملخص مسار ChromaDB (المحاضرات 25، 26، 27):
1. أنشأنا مستندات وقرأناها بـ `DirectoryLoader`.
2. قطعنا النصوص لـ `Chunks` بـ `RecursiveCharacterTextSplitter`.
3. ولدنا متجهات التضمين وحفظناها في `ChromaDB` على القرص الصلب محلياً.
4. حولنا الفهرس لـ `Retriever` يسترجع أدق القطع.
5. ربطنا السياق بنموذج LLM عبر `create_stuff_documents_chain` و `create_retrieval_chain`.

**المحاضرة التالية (28):** سنبني نفس المسار باستخدام لغة تعبير لانج تشين الحديثة (`LCEL`) بأسلوب أنقى وأكثر مرونة!